In [ ]:
import torch, os
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("DEVICE:", "cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


torch: 2.9.0+cu126
cuda available: True
DEVICE: cuda
GPU: Tesla T4


In [ ]:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers scikit-learn pandas numpy tqdm pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 11.4 MB/s eta 0:00:00


In [ ]:
import os, re, json, random, numpy as np, pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pypdf import PdfReader
from transformers import AutoTokenizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| torch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


DEVICE: cuda | torch: 2.9.0+cu126
GPU: Tesla T4


In [ ]:
# ✅ CELDA 3: Carga PDFs (Secundaria + Primaria) y extrae texto (PDFs no escaneados)

import os
from pathlib import Path
from google.colab import drive

from pypdf import PdfReader

# 1) Montar Google Drive
drive.mount('/content/drive')

# 2) Ruta de la carpeta donde están tus PDFs en Drive (CAMBIA SOLO ESTO)
PDF_DIR = "/content/drive/MyDrive/TEXTOS_RED_1"  # <-- EJ: "/content/drive/MyDrive/PDFS_2025"


# Nombres exactos de los archivos que subiste
PDF_SEC = "Prontuario de mis aprendizajes - Educación Secundaria Comunitaria Productiva.pdf"
PDF_PRI = "Prontuario de mis aprendizajes - Educación Primaria Comunitaria Vocacional.pdf"

# NUEVOS
PDF_ACT_6TO_PRI = "Cuaderno de Actividades-6to_primaria-2025.pdf"
PDF_ACT_1RO_PRI = "Cuaderno de Actividades-1ro_primaria-2025.pdf"

PDF_FISICA_SEC = "Solucionario de Fisica Tomo I - Educación Secundaria Comunitaria Productiva [2025].pdf"
PDF_QUIMICA_SEC = "Solucionario de Química Tomo II - Educación Secundaria Comunitaria Productiva [2025].pdf"
PDF_MATEMATICA_SEC = "Solucionario de Matemática Tomo II - Educación Secundaria Comunitaria Productiva [2025].pdf"
PDF_LOGICA_SEC = "Desarrollo del Razonamiento Lógico - Educación Secundaria Comunitaria Productiva [2025].pdf"


PDF_RM_SUPERIOR = "Resolución Ministerial N° 0001-2026 - Normas Generales para la Gestión Educativa 2026 del Subsistema de Educación Superior de Formación Profesional Técnica, Tecnológica y Artística.pdf"
PDF_RM_REGULAR = "Resolución Ministerial N° 0001-2026 - Normas Generales para la Gestión Educativa 2026 del Subsistema de Educación Regular.pdf"
PDF_RM_ALTERNATIVA = "Resolución Ministerial N° 0001-2026 - Normas Generales para la Gestión Educativa 2026 del Subsistema de Educación Alternativa y Especial.pdf"

PDF_1ER_ESC = "1er año de escolaridad - Valores, Espiritualidades y Religiones [2025].pdf"
PDF_4TO_ESC = "4to año de escolaridad - Cosmovisiones, Filosofía y Psicología [2025].pdf"
PDF_5TO_ESC = "5to año de escolaridad [2025].pdf"
PDF_2DO_ESC = "2do año de escolaridad - Técnica Tecnológica General [2025].pdf"
PDF_3ER_ESC_1 = "3er año de escolaridad [2025] (1).pdf"
PDF_6TO_ESC = "6to año de escolaridad [2025].pdf"
PDF_3ER_ESC = "3er año de escolaridad [2025].pdf"


# 3) Función para buscar PDF EN TU CARPETA DE DRIVE (y subcarpetas)
import unicodedata
from pathlib import Path

PDF_DIR = "/content/drive/MyDrive/TEXTOS_RED_1"

def normalize(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii").lower()

def find_pdf(pdf_name, base_dir=PDF_DIR):
    target = normalize(pdf_name)
    for p in Path(base_dir).rglob("*.pdf"):
        if normalize(p.name) == target:
            return str(p)
    raise FileNotFoundError(
        f"No encontré el PDF: {pdf_name}\n"
        f"Busqué dentro de: {base_dir}"
    )



# Buscar y leer PDFs
sec_path = find_pdf(PDF_SEC)
pri_path = find_pdf(PDF_PRI)

# NUEVOS
act_6to_pri_path = find_pdf(PDF_ACT_6TO_PRI)
act_1ro_pri_path = find_pdf(PDF_ACT_1RO_PRI)

fisica_sec_path = find_pdf(PDF_FISICA_SEC)
quimica_sec_path = find_pdf(PDF_QUIMICA_SEC)
matematica_sec_path = find_pdf(PDF_MATEMATICA_SEC)
logica_sec_path = find_pdf(PDF_LOGICA_SEC)


rm_superior_path = find_pdf(PDF_RM_SUPERIOR)
rm_regular_path = find_pdf(PDF_RM_REGULAR)
rm_alternativa_path = find_pdf(PDF_RM_ALTERNATIVA)

esc_1er_path = find_pdf(PDF_1ER_ESC)
esc_4to_path = find_pdf(PDF_4TO_ESC)
esc_5to_path = find_pdf(PDF_5TO_ESC)
esc_2do_path = find_pdf(PDF_2DO_ESC)
esc_3er_1_path = find_pdf(PDF_3ER_ESC_1)
esc_6to_path = find_pdf(PDF_6TO_ESC)
esc_3er_path = find_pdf(PDF_3ER_ESC)


print("PDF Secundaria encontrado en:", sec_path)
print("PDF Primaria encontrado en:", pri_path)


# NUEVOS
print("PDF Cuaderno de Actividades 6to Primaria encontrado en:", act_6to_pri_path)
print("PDF Cuaderno de Actividades 1ro Primaria encontrado en:", act_1ro_pri_path)

print("PDF Solucionario de Física Secundaria encontrado en:", fisica_sec_path)
print("PDF Solucionario de Química Secundaria encontrado en:", quimica_sec_path)
print("PDF Solucionario de Matemática Secundaria encontrado en:", matematica_sec_path)
print("PDF Desarrollo del Razonamiento Lógico Secundaria encontrado en:", logica_sec_path)


print("PDF Resolución Ministerial Educación Superior encontrado en:", rm_superior_path)
print("PDF Resolución Ministerial Educación Regular encontrado en:", rm_regular_path)
print("PDF Resolución Ministerial Educación Alternativa y Especial encontrado en:", rm_alternativa_path)

print("PDF 1er año de escolaridad encontrado en:", esc_1er_path)
print("PDF 4to año de escolaridad encontrado en:", esc_4to_path)
print("PDF 5to año de escolaridad encontrado en:", esc_5to_path)
print("PDF 2do año de escolaridad encontrado en:", esc_2do_path)
print("PDF 3er año de escolaridad (1) encontrado en:", esc_3er_1_path)
print("PDF 6to año de escolaridad encontrado en:", esc_6to_path)
print("PDF 3er año de escolaridad encontrado en:", esc_3er_path)


# Función para extraer texto de un PDF
def extract_text_from_pdf(path):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text

# Extraer textos
full_text_sec = extract_text_from_pdf(sec_path)
full_text_pri = extract_text_from_pdf(pri_path)


# NUEVOS
text_act_6to_pri = extract_text_from_pdf(act_6to_pri_path)
text_act_1ro_pri = extract_text_from_pdf(act_1ro_pri_path)

text_fisica_sec = extract_text_from_pdf(fisica_sec_path)
text_quimica_sec = extract_text_from_pdf(quimica_sec_path)
text_matematica_sec = extract_text_from_pdf(matematica_sec_path)
text_logica_sec = extract_text_from_pdf(logica_sec_path)


text_rm_superior = extract_text_from_pdf(rm_superior_path)
text_rm_regular = extract_text_from_pdf(rm_regular_path)
text_rm_alternativa = extract_text_from_pdf(rm_alternativa_path)

text_1er_esc = extract_text_from_pdf(esc_1er_path)
text_4to_esc = extract_text_from_pdf(esc_4to_path)
text_5to_esc = extract_text_from_pdf(esc_5to_path)
text_2do_esc = extract_text_from_pdf(esc_2do_path)
text_3er_1_esc = extract_text_from_pdf(esc_3er_1_path)
text_6to_esc = extract_text_from_pdf(esc_6to_path)
text_3er_esc = extract_text_from_pdf(esc_3er_path)


# Combinar textos
full_text = (full_text_sec + "\n" + full_text_pri +
text_act_6to_pri + "\n" +
text_act_1ro_pri + "\n" +
text_fisica_sec + "\n" +
text_quimica_sec + "\n" +
text_matematica_sec + "\n" +
text_logica_sec + "\n" +
text_rm_superior + "\n" +
text_rm_regular + "\n" +
text_rm_alternativa + "\n" +
text_1er_esc + "\n" +
text_4to_esc + "\n" +
text_5to_esc + "\n" +
text_2do_esc + "\n" +
text_3er_1_esc + "\n" +
text_6to_esc + "\n" +
text_3er_esc)

print("Caracteres totales combinados extraídos:", len(full_text))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDF Secundaria encontrado en: /content/drive/MyDrive/TEXTOS_RED_1/Prontuario de mis aprendizajes - Educación Secundaria Comunitaria Productiva.pdf
PDF Primaria encontrado en: /content/drive/MyDrive/TEXTOS_RED_1/Prontuario de mis aprendizajes - Educación Primaria Comunitaria Vocacional.pdf
PDF Cuaderno de Actividades 6to Primaria encontrado en: /content/drive/MyDrive/TEXTOS_RED_1/Cuaderno de Actividades-6to_primaria-2025.pdf
PDF Cuaderno de Actividades 1ro Primaria encontrado en: /content/drive/MyDrive/TEXTOS_RED_1/Cuaderno de Actividades-1ro_primaria-2025.pdf
PDF Solucionario de Física Secundaria encontrado en: /content/drive/MyDrive/TEXTOS_RED_1/Solucionario de Fisica Tomo I - Educación Secundaria Comunitaria Productiva [2025].pdf
PDF Solucionario de Química Secundaria encontrado en: /content/drive/MyDrive/TEXTOS_RED_1/Solucionario de Química Tomo II - E

In [ ]:
# ✅ CELDA 4: Limpieza y división en chunks robustos (para PDFs largos)
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def chunk_text(text: str, max_words: int = 260, overlap_words: int = 60):
    # Divide por palabras con overlap para no perder contexto en límites
    words = text.split()
    if len(words) < 30:
        return []

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        c = " ".join(words[start:end]).strip()
        if len(c) >= 120:  # filtro mínimo para evitar basura
            chunks.append(c)
        if end == len(words):
            break
        start = max(0, end - overlap_words)

    return chunks

full_text = clean_text(full_text)
chunks = chunk_text(full_text, max_words=260, overlap_words=60)

print("Total de chunks:", len(chunks))
print("Ejemplo de chunk (primeros 350 chars):\n", chunks[0][:350])


Total de chunks: 10079
Ejemplo de chunk (primeros 350 chars):
 Prontuario de mis aprendizajes Educación Secundaria Comunitaria Productiva Omar Veliz Ramos MINISTRO DE EDUCACIÓN Manuel Eudal Tejerina del Castillo VICEMINISTRO DE EDUCACIÓN REGULAR Delia Yucra Rodas DIRECTORA GENERAL DE EDUCACIÓN SECUNDARIA Equipo de redacción Dirección General de Educación Secundaria Revisión Instituto de Investigaciones Pedagóg


In [ ]:
# ✅ CELDA 5: Define etiquetas finales (ÁREA x DIMENSIÓN)
# Reglas:
# - Áreas/Componentes típicos en secundaria (ajustable luego, pero estas quedan fijas en el modelo)
# - Dimensiones: SABER / HACER / SER / DECIDIR
# - Etiqueta final = "{AREA}__{DIMENSION}"  -> multi-label

AREAS = [
    "MATEMATICA",
    "COMUNICACION_Y_LENGUAJES",
    "CIENCIAS_NATURALES_BIOLOGIA",
    "FISICA",
    "QUIMICA",
    "CIENCIAS_SOCIALES",
    "VALORES_ESPIRITUALIDAD_RELIGIONES",
    "TECNOLOGIA_PRODUCTIVA",
]

DIMENSIONES = ["SABER", "HACER", "SER", "DECIDIR"]

LABELS = [f"{a}__{d}" for a in AREAS for d in DIMENSIONES]

print("Total de etiquetas:", len(LABELS))
print("Primeras 12 etiquetas:", LABELS[:12])
print("Últimas 12 etiquetas:", LABELS[-12:])


Total de etiquetas: 32
Primeras 12 etiquetas: ['MATEMATICA__SABER', 'MATEMATICA__HACER', 'MATEMATICA__SER', 'MATEMATICA__DECIDIR', 'COMUNICACION_Y_LENGUAJES__SABER', 'COMUNICACION_Y_LENGUAJES__HACER', 'COMUNICACION_Y_LENGUAJES__SER', 'COMUNICACION_Y_LENGUAJES__DECIDIR', 'CIENCIAS_NATURALES_BIOLOGIA__SABER', 'CIENCIAS_NATURALES_BIOLOGIA__HACER', 'CIENCIAS_NATURALES_BIOLOGIA__SER', 'CIENCIAS_NATURALES_BIOLOGIA__DECIDIR']
Últimas 12 etiquetas: ['CIENCIAS_SOCIALES__SABER', 'CIENCIAS_SOCIALES__HACER', 'CIENCIAS_SOCIALES__SER', 'CIENCIAS_SOCIALES__DECIDIR', 'VALORES_ESPIRITUALIDAD_RELIGIONES__SABER', 'VALORES_ESPIRITUALIDAD_RELIGIONES__HACER', 'VALORES_ESPIRITUALIDAD_RELIGIONES__SER', 'VALORES_ESPIRITUALIDAD_RELIGIONES__DECIDIR', 'TECNOLOGIA_PRODUCTIVA__SABER', 'TECNOLOGIA_PRODUCTIVA__HACER', 'TECNOLOGIA_PRODUCTIVA__SER', 'TECNOLOGIA_PRODUCTIVA__DECIDIR']


In [ ]:
# ✅ CELDA 6: Auto-etiquetado determinista (reglas) para poder ENTRENAR hoy con tu PDF
# IMPORTANTE:
# - Esto NO es "simulación aleatoria".
# - Es weak-supervision: reglas reproducibles basadas en palabras-clave.
# - En producción, puedes mejorar etiquetado con validación docente (active learning) sin cambiar arquitectura.

AREA_KEYWORDS = {
    "MATEMATICA": [
        # Conceptos generales y Aritmética
        "números y operaciones", "sistemas de numeración", "conjuntos numéricos",
        "números naturales", "números racionales", "números reales",
        "números irracionales", "números primos", "números decimales",
        "número entero", "numeración binaria", "razones", "proporciones",
        "regla de tres", "análisis numérico", "lógica matemática",

        "cuánto es", "cuál es el resultado", "resuelve", "calcula", "hallar",
        "simplifica", "despeja", "demuestra", "fórmula", "problema", "ejercicio",
        "ecuación", "inecuación", "sistema de ecuaciones", "función",
        "gráfica", "tabla de valores", "porcentaje", "fracción", "raíz cuadrada",
        "logaritmo", "derivada", "integral"
        # Álgebra y Cálculo
        "álgebra", "expresiones algebraicas", "ecuaciones de primer grado",
        "ecuaciones cuadráticas", "sistemas de ecuaciones", "desigualdades",
        "inecuaciones", "factorización", "fracciones algebraicas",
        "exponentes", "radicales", "números complejos", "sucesiones",
        "progresiones", "análisis matemático", "cálculo infinitesimal",
        "límites", "continuidad", "derivada", "integrales", "álgebra lineal",
        # Geometría y Trigonometría
        "geometría", "plano cartesiano", "triángulos", "semejanza",
        "áreas", "perímetros", "formas tridimensionales", "geometría analítica",
        "línea recta", "circunferencia", "parábola", "elipse", "hipérbola",
        "trigonometría", "identidades trigonométricas", "análisis vectorial",
        # Estadística y Otros
        "estadística", "probabilidad", "combinatoria", "matemática financiera",
        "teoría de conjuntos", "automatización", "inteligencia artificial"
    ],
    "COMUNICACION_Y_LENGUAJES": [
        # Lingüística y Gramática
        "lenguaje oral", "lenguaje escrito", "signos sonoros", "fonología",
        "fonética", "habla", "morfología gramatical", "verbos regulares",
        "sustantivos", "adjetivos", "neologismos", "palabras homógrafas",
        "palabras multiformes", "polisémicas", "definiciones", "mayúsculas",

        "resume", "resumen", "explica con tus palabras", "idea principal",
        "tema central", "argumenta", "opina", "ensayo", "redacción",
        "sinónimo", "antónimo", "ortografía", "acentuación", "puntuación",
        "comprensión lectora", "análisis de texto", "coherencia", "cohesión"
        # Tipos de texto y Comunicación
        "medios de comunicación social", "masas", "géneros periodísticos",
        "textos orales", "cómic", "comunicación efectiva", "monografía",
        "investigación", "referencias bibliográficas", "justificación",
        "objetivos", "mapas mentales", "mapa conceptual", "trabalenguas"
    ],
    "CIENCIAS_NATURALES_BIOLOGIA": [
        # Ramas de la Biología
        "biología", "botánica", "zoología", "ecología", "microbiología",
        "virología", "genética", "entomología", "ornitología", "herpetología",
        "bacteriología", "protozoología", "pteridología", "ficología",
        "ictiología", "biomatemáticas", "biogeografía", "bioquímica",
        "biofísica", "bioética",
        # Conceptos Biológicos
        "vida", "organismo", "célula", "moléculas orgánicas", "nutrientes",
        "procesos metabólicos", "evolución", "crecimiento", "origen",
        "distribución", "ecosistemas", "naturaleza",

        "ecosistema", "cadena alimenticia", "fotosíntesis", "respiración celular",
        "ADN", "ARN", "cromosomas", "mitosis", "meiosis", "gen", "mutación",
        "biodiversidad", "hábitat", "población", "comunidad", "bioma",
        "cuerpo humano", "órganos", "sistema digestivo", "sistema respiratorio",
        "enfermedad", "vacuna"
    ],
    "FISICA": [
        # Ramas de la Física
        "física clásica", "física moderna", "física contemporánea", "mecánica",
        "termodinámica", "movimiento ondulatorio", "óptica", "electromagnetismo",
        "relatividad", "mecánica cuántica", "física de partículas", "gravitación",
        "dinámica no lineal", "sistemas complejos", "nanofísica",
        # Conceptos y Mediciones
        "materia", "energía", "fenómenos naturales", "mediciones", "magnitudes",
        "cantidad patrón", "cinemática", "dinámica", "velocidad", "tiempo",
        "espacio", "vector", "campo vectorial", "fuerza",

        "movimiento parabólico", "tiro parabólico", "MRU", "MRUV",
        "aceleración", "velocidad", "desplazamiento", "trayectoria",
        "leyes de newton", "gravedad", "masa", "peso", "trabajo", "potencia",
        "energía cinética", "energía potencial", "presión", "densidad",
        "carga eléctrica", "corriente eléctrica", "voltaje", "resistencia"
    ],
    "QUIMICA": [
        # General y Ramas
        "química general", "estructura de la materia", "composición",
        "transformaciones", "elementos naturales",
        # Aplicaciones y Contexto
        "fármacos", "plaguicidas", "armas químicas", "compuestos",
        "análisis empírico", "contexto inorgánico",

        "átomo", "molécula", "tabla periódica", "enlace químico",
        "reacción", "ecuación química", "balanceo", "ácido", "base",
        "ph", "oxidación", "reducción", "mezcla", "solución",
        "concentración", "moles", "estequiometría"
    ],
    "CIENCIAS_SOCIALES": [
        # Disciplinas
        "historia", "antropología", "sociología", "ciencia política",
        "economía", "lingüística", "psicología", "derecho", "geografía",
        "demografía", "arqueología", "ciencias de la educación",

        "independencia", "colonia", "virreinato", "revolución", "batalla",
        "libertadores", "simón bolívar", "san martín", "sucre",
        "napoleón", "ilustración", "esclavitud", "castas", "criollos",
        "imperio", "república", "constitución", "estado", "gobierno",
        "democracia", "ciudadanía", "derechos", "economía", "sociedad",
        "historia de bolivia", "guerra", "tratado", "frontera", "territorio"
        # Conceptos Políticos y Sociales
        "estado plurinacional", "constitución política", "democracia",
        "gobierno", "poder político", "conflictos sociales", "instituciones",
        "territorialidad", "función pública", "ley 450", "vulnerabilidad",
        "naciones y pueblos indígena originarios", "aislamiento voluntario",
        # Economía
        "producción", "distribución", "consumo", "bienes y servicios",
        "producto interno bruto (pib)", "desarrollo económico", "comercio internacional"
    ],
    "VALORES_ESPIRITUALIDAD_RELIGIONES": [
        # Principios Andinos y Constitucionales
        "vivir bien", "suma qamaña", "ñandereko", "teko kavi", "ivi maraei",
        "qhapaj ñan", "ama qhilla", "ama llulla", "ama suwa",
        # Valores Éticos
        "unidad", "igualdad", "inclusión", "dignidad", "libertad",
        "solidaridad", "reciprocidad", "respeto", "complementariedad",
        "armonía", "transparencia", "equilibrio", "bienestar común",
        "responsabilidad", "justicia social", "vida armoniosa",

        "ética", "moral", "valores", "convivencia", "responsabilidad social",
        "respeto a la madre tierra", "pachamama", "solidaridad", "empatía",
        "justicia", "equidad", "derechos humanos", "interculturalidad"
    ],

    "TECNOLOGIA_PRODUCTIVA": [
        "inteligencia artificial", "programación", "automatización", "robot",
        "redes neuronales", "chatbot", "proyecto", "plan sectorial",
        "necesidad", "solución de problemas", "intervención", "investigación",

         "software", "hardware", "computadora", "internet", "wifi",
        "aplicación", "sistema", "base de datos", "api", "backend", "frontend",
        "algoritmo", "modelo", "entrenamiento", "dataset", "red neuronal",
        "machine learning", "deep learning", "inferencias", "predicción"
    ],
}

DIM_KEYWORDS = {
    "SABER": [
        "definir", "reconocer", "identificar", "conocer", "describir",
        "explicar", "comprender", "interpretar", "analizar", "estudiar",
        "adquirir conocimientos", "profundizar", "entender", "teorizar",

        "qué es", "quién es", "quién fue", "cuándo", "dónde", "por qué",
        "para qué", "explica", "describe", "menciona", "enumera",
        "resume", "definición", "concepto", "características"
    ],
    "HACER": [
        "resolver", "aplicar", "elaborar", "construir", "desarrollar",
        "realizar", "calcular", "producir", "experimentar", "investigar",
        "redactar", "medir", "comparar", "transformar", "ejecutar",
        "diseñar", "practicar", "utilizar",

         "haz", "hace", "realiza", "crea", "escribe", "redacta", "dibuja",
        "calcula", "resuelve", "aplica", "demuestra", "completa",
        "ejercicio", "actividad", "práctica", "pasos", "procedimiento"
    ],
    "SER": [
        "valorar", "respetar", "asumir", "convivir", "colaborar",
        "responsable", "solidario", "ético", "actitud", "reciprocidad",
        "complementariedad", "integridad", "consciente", "dignidad",
        "inclusión", "equidad",

        "reflexiona", "valora", "opina con respeto", "convivencia",
        "empatía", "solidaridad", "responsabilidad", "ética",
        "compromiso", "conciencia", "identidad", "respeto"
    ],
    "DECIDIR": [
        "decidir", "elegir", "proponer", "planificar", "argumentar",
        "evaluar", "seleccionar", "priorizar", "comprometer", "promover",
        "transformar la realidad", "satisfacer necesidades", "vincular",
        "integrar", "participar", "solucionar",

        "propón", "elige", "decide", "plantea", "argumenta", "justifica",
        "toma una postura", "qué harías", "qué decisión", "solución",
        "plan", "planifica", "recomienda", "evalúa", "concluye"
    ],
}

MIN_AREA_HITS = 2      # exige evidencia para asignar área
MAX_DIMS = 2           # evita explotar etiquetas
FALLBACK = "CIENCIAS_SOCIALES__SABER"

# --- FIX OBLIGATORIO: definir infer_areas / infer_dims ---
def infer_areas(text: str):
    t = (text or "").lower().strip()

    # Heurística mínima: en texto corto (pregunta) pide menos evidencia
    # (NO cambia MIN_AREA_HITS global; solo ajusta localmente)
    local_min_hits = 1 if len(t) < 140 else MIN_AREA_HITS

    hits = []
    for area, kws in AREA_KEYWORDS.items():
        score = 0
        for kw in kws:
            if kw in t:
                score += 1
        if score >= local_min_hits:
            hits.append((area, score))

    # Si no hay hits suficientes, fallback controlado (NO lista vacía)
    if not hits:
        # fallback devuelve solo el área base (sin dimensión)
        return [FALLBACK.split("__")[0]]

    hits.sort(key=lambda x: x[1], reverse=True)
    return [a for a, _ in hits]


def infer_dims(text: str):
    t = (text or "").lower().strip()
    dims = []
    for d, kws in DIM_KEYWORDS.items():
        if any(kw in t for kw in kws):
            dims.append(d)

    # Si no detecta dimensión (preguntas cortas), forzar SABER como default
    if not dims:
        return ["SABER"]

    # Respeta tu MAX_DIMS sin cambiar lógica
    return dims[:MAX_DIMS]
# --- FIN FIX ---


def label_area(text: str):
    areas = infer_areas(text)          # tu función
    return areas[0] if areas else None

def label_dims(text: str):
    dims = infer_dims(text)            # tu función
    return dims if dims else None

rows = []
for c in tqdm(chunks, desc="Etiquetando chunks"):
    a = label_area(c)
    d = label_dims(c)
    if a is None or d is None:
        continue  # fuera basura
    rows.append({"text": c, "area": a, "dims": d})

df = pd.DataFrame(rows).reset_index(drop=True)
print(df.shape)
print(df.head())



Etiquetando chunks: 100%|██████████| 10079/10079 [00:07<00:00, 1322.04it/s]

(10079, 3)
                                                text  \
0  Prontuario de mis aprendizajes Educación Secun...   
1  general DE MIS APRENDIZAJES Geometría analític...   
2  Óxidos ácidos o anhídridos ......................   
3  583 Articulaciones ..............................   
4  lenguajes Comunicación La comunicación ..........   

                          area          dims  
0                   MATEMATICA  [HACER, SER]  
1                   MATEMATICA       [HACER]  
2  CIENCIAS_NATURALES_BIOLOGIA         [SER]  
3            CIENCIAS_SOCIALES  [HACER, SER]  
4     COMUNICACION_Y_LENGUAJES  [HACER, SER]  


In [ ]:
# Celda 7

TOKENIZER_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

MAX_LEN = 256
BATCH_SIZE = 32

# Área (single label)
area_to_id = {a:i for i,a in enumerate(AREAS)}
y_area = np.array([area_to_id[a] for a in df["area"].tolist()])

# Dimensiones (multi-label 4)
mlb_dim = MultiLabelBinarizer(classes=DIMENSIONES)
y_dim = mlb_dim.fit_transform(df["dims"].values)

X_train, X_val, ya_train, ya_val, yd_train, yd_val = train_test_split(
    df["text"].tolist(), y_area, y_dim, test_size=0.2, random_state=SEED
)

enc_train = tokenizer(X_train, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
enc_val   = tokenizer(X_val,   truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")

class DualDataset(Dataset):
    def __init__(self, enc, ya, yd):
        self.ids = enc["input_ids"]
        self.att = enc["attention_mask"]
        self.ya = torch.tensor(ya).long()
        self.yd = torch.tensor(yd).float()

    def __len__(self): return self.ids.size(0)

    def __getitem__(self, idx):
        return {"input_ids": self.ids[idx], "attention_mask": self.att[idx], "y_area": self.ya[idx], "y_dim": self.yd[idx]}

train_ds = DualDataset(enc_train, ya_train, yd_train)
val_ds   = DualDataset(enc_val,   ya_val,   yd_val)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print("Train:", len(train_ds), "Val:", len(val_ds))



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Train: 8063 Val: 2016


In [ ]:
# ✅ CELDA 8: Modelo LSTM multi-head + entrenamiento robusto (SER/DECIDIR raras)

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

VOCAB_SIZE = tokenizer.vocab_size
PAD_ID = tokenizer.pad_token_id
NUM_AREAS = len(AREAS)
NUM_DIMS  = len(DIMENSIONES)

EMB_DIM = 256
HIDDEN = 256

class MultiHeadPedagogicalLSTM(nn.Module):
    def __init__(self, vocab_size, pad_id, emb_dim, hidden_dim, num_areas, num_dims):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.3)

        self.fc_area = nn.Linear(hidden_dim * 2, num_areas)  # softmax
        self.fc_dim  = nn.Linear(hidden_dim * 2, num_dims)   # sigmoid multi-label

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)      # [B,T,E]
        out, _ = self.lstm(x)              # [B,T,2H]

        mask = attention_mask.unsqueeze(-1).float()  # [B,T,1]
        out = out * mask
        pooled = out.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)  # [B,2H]

        pooled = self.dropout(pooled)

        logits_area = self.fc_area(pooled)  # [B,num_areas]
        logits_dim  = self.fc_dim(pooled)   # [B,num_dims]
        return logits_area, logits_dim

model = MultiHeadPedagogicalLSTM(VOCAB_SIZE, PAD_ID, EMB_DIM, HIDDEN, NUM_AREAS, NUM_DIMS).to(DEVICE)

# ✅ PESOS para ÁREA (desbalance)
area_counts = np.bincount(ya_train, minlength=NUM_AREAS).astype(np.float32)
area_weights = (area_counts.sum() / (area_counts + 1e-6))
area_weights = torch.tensor(area_weights, dtype=torch.float32).to(DEVICE)
loss_area = nn.CrossEntropyLoss(weight=area_weights)

# ✅ pos_weight + Focal para DIM (SER/DECIDIR raras)
pos = yd_train.sum(axis=0)                 # positivos por dim
neg = yd_train.shape[0] - pos
pos_weight_dim = torch.tensor(neg / (pos + 1e-6), dtype=torch.float32).to(DEVICE)

class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)  # [B,num_dims]
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        focal = (1 - p_t).pow(self.gamma)
        return (focal * bce).mean()

loss_dim = FocalBCEWithLogitsLoss(gamma=2.0, pos_weight=pos_weight_dim)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)

EPOCHS = 20
print("Entrenando multi-head...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0.0

    for b in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS}"):
        ids = b["input_ids"].to(DEVICE)
        att = b["attention_mask"].to(DEVICE)
        y_a = b["y_area"].to(DEVICE)         # [B]
        y_d = b["y_dim"].to(DEVICE)          # [B,4]

        optimizer.zero_grad()
        logits_a, logits_d = model(ids, att)

        la = loss_area(logits_a, y_a)
        ld = loss_dim(logits_d, y_d)

        loss = la + 1.0 * ld  # ✅ peso 1:1 estable (no invento nada)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item()

    print(f"Epoch {epoch} | Loss: {total/len(train_dl):.4f}")


Entrenando multi-head...


Epoch 1/20: 100%|██████████| 252/252 [00:09<00:00, 25.61it/s]


Epoch 1 | Loss: 2.1266


Epoch 2/20: 100%|██████████| 252/252 [00:08<00:00, 28.88it/s]


Epoch 2 | Loss: 1.8513


...

Epoch 18 | Loss: 0.5700


Epoch 19/20: 100%|██████████| 252/252 [00:09<00:00, 27.48it/s]


Epoch 19 | Loss: 0.5394


Epoch 20/20: 100%|██████████| 252/252 [00:09<00:00, 27.26it/s]

Epoch 20 | Loss: 0.4993


In [ ]:
# ✅ CELDA 9: Calibración threshold para DIM (solo SABER/HACER) + export calibration.json

import numpy as np
import json
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, classification_report

def eval_dim_threshold(th, top_k=2):
    model.eval()
    all_probs, all_true = [], []

    with torch.no_grad():
        for b in val_dl:
            ids = b["input_ids"].to(DEVICE, non_blocking=True)
            att = b["attention_mask"].to(DEVICE, non_blocking=True)
            y_true = b["y_dim"].cpu().numpy()  # [N,4]

            _, logits_d = model(ids, att)
            probs_d = torch.sigmoid(logits_d).cpu().numpy()  # [N,4]

            all_probs.append(probs_d)
            all_true.append(y_true)

    y_prob = np.vstack(all_probs)
    y_true = np.vstack(all_true)

    # ✅ umbral SOLO para SABER/HACER
    y_pred = np.zeros_like(y_true)
    y_pred[:, 0] = (y_prob[:, 0] >= th).astype(int)  # SABER
    y_pred[:, 1] = (y_prob[:, 1] >= th).astype(int)  # HACER

    # ✅ fuerza mínimo 1 de las dos (si ninguna activó)
    for i in range(y_pred.shape[0]):
        if (y_pred[i, 0] + y_pred[i, 1]) == 0:
            j = int(np.argmax(y_prob[i, :2]))  # solo entre saber/hacer
            y_pred[i, j] = 1

    # Métricas SOLO sobre saber/hacer
    y_true2 = y_true[:, :2]
    y_pred2 = y_pred[:, :2]

    return {
        "th": float(th),
        "top_k": int(top_k),
        "f1_micro": float(f1_score(y_true2, y_pred2, average="micro", zero_division=0)),
        "f1_macro": float(f1_score(y_true2, y_pred2, average="macro", zero_division=0)),
        "precision_macro": float(precision_score(y_true2, y_pred2, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true2, y_pred2, average="macro", zero_division=0)),
        "hamming": float(hamming_loss(y_true2, y_pred2)),
        "avg_dims_per_text": float(y_pred2.sum(axis=1).mean()),
        "report": classification_report(y_true2, y_pred2, target_names=DIMENSIONES[:2], zero_division=0)
    }

best = None
for th in np.arange(0.20, 0.61, 0.05):
    m = eval_dim_threshold(float(th), top_k=2)
    print({k: m[k] for k in ["th","f1_micro","f1_macro","hamming","avg_dims_per_text"]})
    if best is None or m["f1_micro"] > best["f1_micro"]:
        best = m

print("\n✅ BEST:", {k: best[k] for k in ["th","f1_micro","f1_macro","hamming","avg_dims_per_text"]})
print("\nReporte calibración SABER/HACER:\n", best["report"])

with open("calibration.json", "w", encoding="utf-8") as f:
    json.dump({
        "dim_threshold": best["th"],
        "dim_top_k": 2,
        "areas": AREAS,
        "dims": DIMENSIONES
    }, f, ensure_ascii=False, indent=2)

print("✅ Exportado: calibration.json")


{'th': 0.2, 'f1_micro': 0.7536321483771252, 'f1_macro': 0.7411602193148683, 'hamming': 0.39533730158730157, 'avg_dims_per_text': 2.0}
{'th': 0.25, 'f1_micro': 0.7536321483771252, 'f1_macro': 0.7411602193148683, 'hamming': 0.39533730158730157, 'avg_dims_per_text': 2.0}
{'th': 0.3, 'f1_micro': 0.7539817535178599, 'f1_macro': 0.7415096945951376, 'hamming': 0.39459325396825395, 'avg_dims_per_text': 1.9985119104385376}
{'th': 0.35, 'f1_micro': 0.760369384880263, 'f1_macro': 0.7478153022909471, 'hamming': 0.37971230158730157, 'avg_dims_per_text': 1.9598214626312256}
{'th': 0.39999999999999997, 'f1_micro': 0.7658411064651013, 'f1_macro': 0.7525408420717346, 'hamming': 0.3611111111111111, 'avg_dims_per_text': 1.875}
{'th': 0.44999999999999996, 'f1_micro': 0.7556084296397009, 'f1_macro': 0.7435938143943651, 'hamming': 0.3566468253968254, 'avg_dims_per_text': 1.7093254327774048}
{'th': 0.49999999999999994, 'f1_micro': 0.7223812249570692, 'f1_macro': 0.7113484292537604, 'hamming': 0.3608630952380

In [ ]:
# ✅ CELDA 10: Inferencia FINAL (robusta)
# - Área: reglas (porque en tus PDFs hay encabezados como "FÍSICA", "CIENCIAS SOCIALES", etc.)
# - Dim: LSTM
# - SER/DECIDIR: solo si keywords o prob muy alta (gating)

import json
import numpy as np

with open("calibration.json", "r", encoding="utf-8") as f:
    calib = json.load(f)

DIM_TH = float(calib["dim_threshold"])
# MAX_DIMS_OUT = int(calib["dim_top_k"])
MAX_DIMS_OUT = 4


# thresholds más estrictos para raras
TH_SER = 0.65
TH_DECIDIR = 0.75

SER_KWS = ["valorar","respeto","convivir","solidaridad","ético","ética","actitud","equidad","dignidad","inclusión","responsable"]
DEC_KWS = ["decidir","elegir","proponer","planificar","priorizar","comprometer","promover","argumentar","evaluar","seleccionar"]

def has_kw(text, kws):
    t = text.lower()
    return any(k in t for k in kws)

def rule_area(text: str):
    t = text.upper()
    # 1) prioridad por encabezados del PDF
    for a in AREAS:
        if a.replace("_", " ") in t:
            return a
    # 2) fallback a tus reglas existentes (CELDA 6)
    areas = infer_areas(text)
    return areas[0] if areas else "CIENCIAS_SOCIALES"

def pick_dims(text, prob_d):
    # prob_d en orden DIMENSIONES = ["SABER","HACER","SER","DECIDIR"]
    out = []

    # SABER/HACER con threshold calibrado
    if prob_d[0] >= DIM_TH: out.append(0)
    if prob_d[1] >= DIM_TH: out.append(1)

    # SER/DECIDIR con gating (keywords o confianza muy alta)
    if has_kw(text, SER_KWS) or prob_d[2] >= TH_SER:
        out.append(2)
    if has_kw(text, DEC_KWS) or prob_d[3] >= TH_DECIDIR:
        out.append(3)

    if len(out) == 0:
        out = [int(np.argmax(prob_d[:2]))]  # mínimo saber/hacer

    # máximo K dims
    out = sorted(set(out), key=lambda i: prob_d[i], reverse=True)[:MAX_DIMS_OUT]
    return out

def predict_final(text: str):
    model.eval()

    area = rule_area(text)

    enc = tokenizer(text, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    ids = enc["input_ids"].to(DEVICE)
    att = enc["attention_mask"].to(DEVICE)

    with torch.no_grad():
        _, logits_d = model(ids, att)
        prob_d = torch.sigmoid(logits_d).cpu().numpy()[0]  # [4]

    dims_idx = pick_dims(text, prob_d)
    dims = [DIMENSIONES[i] for i in dims_idx]

    labels = [{"label": f"{area}__{d}", "score": float(prob_d[DIMENSIONES.index(d)])} for d in dims]
    return {"area": area, "labels": labels}

texto = input("Pega aquí un texto:\n")
print(json.dumps(predict_final(texto), ensure_ascii=False, indent=2))


Pega aquí un texto:
 Vocabulario en evolución, el vocabulario de una lengua está en  constante cambio pues con el pasar del tiempo, se incorporan nuevas  palabras y se desechan otras.  Ejemplo: Debido a la influencia de la tecnología y la cultura mundial,  términos como smartphone y selfie han sido incorporados al vocabulario  de muchos idiomas. b) Cambios gramaticales y fonéticos, son las modificaciones en las  reglas y estructuras del lenguaje, como la sintaxis de las oraciones,  la formación de plurales y la conjugación de verbos. Por otro lado, los  cambios fonéticos se refieren a los cambios en la pronunciación de los  sonidos del lenguaje.  Por ejemplo, Sustitución del vosotros por ustedes en gran parte de  América Latina.  c) Cambios en el significado de la palabra, el significado de las palabras  cambia con el tiempo, este proceso se llama cambio semántico.  Por ejemplo, antiguamente, el término azafata se refería a una “criada  de la reina que se encargaba de los vestidos y jo

In [ ]:
# ✅ CELDA 11: Exporta modelo + labels + tokenizer + config (MULTI-HEAD REAL)

EXPORT_DIR = "/content/model_export_red_1_LSTM"
os.makedirs(EXPORT_DIR, exist_ok=True)

model_path = os.path.join(EXPORT_DIR, "pedagogical_lstm.pt")
labels_path = os.path.join(EXPORT_DIR, "labels.json")
tok_path   = os.path.join(EXPORT_DIR, "tokenizer_name.txt")
cfg_path   = os.path.join(EXPORT_DIR, "config.json")

# pesos
torch.save(model.state_dict(), model_path)

# labels (combinaciones AREA__DIM)
with open(labels_path, "w", encoding="utf-8") as f:
    json.dump(LABELS, f, ensure_ascii=False, indent=2)

# tokenizer
with open(tok_path, "w", encoding="utf-8") as f:
    f.write(TOKENIZER_NAME)

# config (usa tu mejor threshold real)
cfg = {
    "model_type": "lstm_multihead",
    "vocab_size": int(tokenizer.vocab_size),
    "pad_token_id": int(tokenizer.pad_token_id),
    "emb_dim": int(EMB_DIM),
    "hidden_dim": int(HIDDEN),
    "max_len": int(MAX_LEN),

    # clases
    "areas": AREAS,
    "dims": DIMENSIONES,
    "labels": LABELS,

    # calibración (tu BEST)
    "threshold_default": 0.20,
    "top_areas": 3,
    "top_k_combos": 8,
    "max_active_combos": 12
}

with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

print("✅ Export listo en:", EXPORT_DIR)
print("Archivos:", os.listdir(EXPORT_DIR))


✅ Export listo en: /content/model_export_red_1_LSTM
Archivos: ['labels.json', 'tokenizer_name.txt', 'pedagogical_lstm.pt', 'config.json']


In [ ]:
# ✅ CELDA 12: Loader + inferencia FINAL (MULTI-HEAD, igual a tu entrenamiento)

import os, json
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer

# ---------- Modelo MULTI-HEAD ----------
class PedagogicalLSTMMultiHead(nn.Module):
    def __init__(self, vocab_size, pad_token_id, emb_dim, hidden_dim, num_areas, num_dims):
        super().__init__()
        # IMPORTANTE: nombre "embedding" para que coincida con tu state_dict
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_token_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.2)

        # IMPORTANTE: nombres fc_area y fc_dim para que coincidan con tu state_dict
        self.fc_area = nn.Linear(hidden_dim * 2, num_areas)
        self.fc_dim  = nn.Linear(hidden_dim * 2, num_dims)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)          # [B,T,E]
        out, _ = self.lstm(x)                  # [B,T,2H]

        mask = attention_mask.unsqueeze(-1).float()
        out = out * mask
        pooled = out.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)  # masked mean
        pooled = self.dropout(pooled)

        logits_area = self.fc_area(pooled)     # [B,num_areas]
        logits_dim  = self.fc_dim(pooled)      # [B,num_dims]
        return logits_area, logits_dim

# ---------- Loader ----------
def load_classifier(export_dir: str, device: str = "cpu"):
    cfg = json.load(open(os.path.join(export_dir, "config.json"), encoding="utf-8"))
    tok_name = open(os.path.join(export_dir, "tokenizer_name.txt"), encoding="utf-8").read().strip()

    tokenizer = AutoTokenizer.from_pretrained(tok_name)

    areas = cfg["areas"]
    dims  = cfg["dims"]

    model = PedagogicalLSTMMultiHead(
        vocab_size=cfg["vocab_size"],
        pad_token_id=cfg["pad_token_id"],
        emb_dim=cfg["emb_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_areas=len(areas),
        num_dims=len(dims)
    )

    sd = torch.load(os.path.join(export_dir, "pedagogical_lstm.pt"), map_location=device)
    model.load_state_dict(sd, strict=True)
    model.to(device)
    model.eval()

    return model, tokenizer, cfg

# ---------- Utilidades ----------
def build_combo_scores(area_probs, dim_probs, areas, dims):
    """
    Construye scores para todas las combinaciones AREA__DIM como producto:
    score(area__dim) = area_prob * dim_prob
    (estable y consistente para pasar a Red 2)
    """
    combos = []
    for ai, a in enumerate(areas):
        for di, d in enumerate(dims):
            combos.append((f"{a}__{d}", float(area_probs[ai] * dim_probs[di])))
    combos.sort(key=lambda x: -x[1])
    return combos

# ---------- Inferencia FINAL ----------
def classify_text(model, tokenizer, cfg, text: str, threshold: float | None = None, device: str = "cpu"):
    thr = cfg["threshold_default"] if threshold is None else float(threshold)

    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=cfg["max_len"],
        return_tensors="pt"
    )
    ids = enc["input_ids"].to(device)
    att = enc["attention_mask"].to(device)

    with torch.no_grad():
        logits_area, logits_dim = model(ids, att)
        #area_probs = torch.sigmoid(logits_area).cpu().numpy()[0]
        area_probs = torch.softmax(logits_area, dim=-1).cpu().numpy()[0]


        dim_probs  = torch.sigmoid(logits_dim).cpu().numpy()[0]

    areas = cfg["areas"]
    dims  = cfg["dims"]

    # TOP 3 áreas
    area_rank = sorted(
        [{"area": areas[i], "score": float(area_probs[i])} for i in range(len(areas))],
        key=lambda x: -x["score"]
    )
    areas_top3 = area_rank[:cfg["top_areas"]]

    # dims probs (4)
    dims_out = {dims[i]: float(dim_probs[i]) for i in range(len(dims))}

    # combinaciones (AREA__DIM) como producto
    combo_scores = build_combo_scores(area_probs, dim_probs, areas, dims)

    # TOP combos
    top_combos = [{"label": c[0], "score": c[1]} for c in combo_scores[:cfg["top_k_combos"]]]

    # combos activas por threshold (limitadas)
    active = [{"label": c[0], "score": c[1]} for c in combo_scores if c[1] >= thr]
    if not active:
        active = [{"label": combo_scores[0][0], "score": combo_scores[0][1]}]
    active = active[:cfg["max_active_combos"]]

    return {
        "threshold": thr,
        "areas_top": areas_top3,
        "dims_probs": dims_out,
        "combo_top": top_combos,
        "active": active
    }

# ---------- Mini test ----------
export_dir = "/content/model_export_red_1_LSTM"
device = "cuda" if torch.cuda.is_available() else "cpu"

clf_model, clf_tok, clf_cfg = load_classifier(export_dir, device=device)
print("✅ Loader OK | areas:", len(clf_cfg["areas"]), "| dims:", len(clf_cfg["dims"]))

test_text = "Movimiento parabólico, aplicar fórmulas y explicar el procedimiento."
out = classify_text(clf_model, clf_tok, clf_cfg, test_text, device=device)
print(json.dumps(out, ensure_ascii=False, indent=2))


✅ Loader OK | areas: 8 | dims: 4
{
  "threshold": 0.2,
  "areas_top": [
    {
      "area": "QUIMICA",
      "score": 0.7375130653381348
    },
    {
      "area": "CIENCIAS_NATURALES_BIOLOGIA",
      "score": 0.17198659479618073
    },
    {
      "area": "MATEMATICA",
      "score": 0.05121979862451553
    }
  ],
  "dims_probs": {
    "SABER": 0.5193095803260803,
    "HACER": 0.549694299697876,
    "SER": 0.5121142268180847,
    "DECIDIR": 0.48972928524017334
  },
  "combo_top": [
    {
      "label": "QUIMICA__HACER",
      "score": 0.4054067134857178
    },
    {
      "label": "QUIMICA__SABER",
      "score": 0.38299760222435
    },
    {
      "label": "QUIMICA__SER",
      "score": 0.3776909410953522
    },
    {
      "label": "QUIMICA__DECIDIR",
      "score": 0.36118173599243164
    },
    {
      "label": "CIENCIAS_NATURALES_BIOLOGIA__HACER",
      "score": 0.0945400521159172
    },
    {
      "label": "CIENCIAS_NATURALES_BIOLOGIA__SABER",
      "score": 0.0893142893910408


In [ ]:
!zip -r model_export_red_1_LSTM.zip /content/model_export_red_1_LSTM

  adding: content/model_export_red_1_LSTM/ (stored 0%)
  adding: content/model_export_red_1_LSTM/labels.json (deflated 79%)
  adding: content/model_export_red_1_LSTM/tokenizer_name.txt (stored 0%)
  adding: content/model_export_red_1_LSTM/pedagogical_lstm.pt (deflated 7%)
  adding: content/model_export_red_1_LSTM/config.json (deflated 73%)


In [ ]:
from google.colab import files
files.download("model_export_red_1_LSTM.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>